# Mamba-3 vs Transformer on Bangla — Colab driver

Run cells top to bottom. Every long step is resumable — if the session dies,
reconnect, re-run Setup, and re-run the step you were on; it picks up where it left off.

**Pipeline:** setup → tokenizer → pretokenize → param check → profile → toy runs → main runs → eval.

In [ ]:
# === 1. Setup (run on every session) ===
import os, torch
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/sahilaf/Mamba3_Bangla_Case_Study.git'
DRIVE = '/content/drive/MyDrive/mamba3_bn'
DATA = f'{DRIVE}/bn_data'
RUNS = f'{DRIVE}/runs'

# HF Hub robustness: big FineWeb-2 shards blow past the default 10s read timeout.
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
# HF token from Colab Secrets (left sidebar 🔑 -> add HF_TOKEN). Never hard-code it.
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab Secrets")
except Exception:
    print("No HF_TOKEN secret set (optional; add one via the 🔑 panel to raise rate limits)")

if not os.path.exists('/content/repo'):
    !git clone $REPO_URL /content/repo
%cd /content/repo
!git pull
!mkdir -p $DATA $RUNS

In [ ]:
# === 2. Dependencies (run on every session) ===
# mamba-ssm ships CUDA kernels. Prebuilt wheels exist only for specific torch
# minors; Colab sometimes ships a newer torch with no wheel. We pull a matching
# PREBUILT wheel from GitHub Releases (seconds) and cache it to Drive. If Colab's
# torch is unsupported, we pin torch to the nearest supported minor and ask for a
# one-time runtime restart -- still no source build.
!pip -q install sentencepiece datasets einops hf_transfer

import os, glob, sys, subprocess, urllib.request, torch

WHEELS = f'{DRIVE}/wheels'
os.makedirs(WHEELS, exist_ok=True)
MAMBA_VER = "2.3.2.post1"
SUPPORTED_TORCH = ["2.6", "2.7", "2.8", "2.10"]   # minors with published wheels


def _pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)


def _nearest(tmm):
    cur = tuple(int(x) for x in tmm.split("."))
    return min(SUPPORTED_TORCH, key=lambda s: abs((tuple(int(x) for x in s.split("."))[1]) - cur[1]))


def ensure_mamba():
    cached = glob.glob(f"{WHEELS}/mamba_ssm*.whl")
    if cached:
        print("installing cached wheel:", os.path.basename(cached[0]))
        _pip("--no-deps", cached[0]); return

    py = f"cp{sys.version_info.major}{sys.version_info.minor}"
    tmm = ".".join(torch.__version__.split("+")[0].split(".")[:2])
    cu = f"cu{torch.version.cuda.split('.')[0]}"
    abi = "TRUE" if torch._C._GLIBCXX_USE_CXX11_ABI else "FALSE"

    if tmm not in SUPPORTED_TORCH:
        tgt = _nearest(tmm)
        print(f"torch {tmm} has no mamba wheel; pinning torch=={tgt}.* ...")
        _pip(f"torch=={tgt}.*")
        print("\n*** RESTART THE RUNTIME NOW: Runtime > Restart session, "
              "then re-run cells 1 and 2. ***")
        raise SystemExit("restart required after torch downgrade")

    whl = f"mamba_ssm-{MAMBA_VER}+{cu}torch{tmm}cxx11abi{abi}-{py}-{py}-linux_x86_64.whl"
    url = f"https://github.com/state-spaces/mamba/releases/download/v{MAMBA_VER}/{whl}"
    dst = f"{WHEELS}/{whl}"
    print(f"env: {py} torch{tmm} {cu} abi{abi}\ntrying prebuilt: {url}")
    try:
        urllib.request.urlretrieve(url, dst)
        _pip("--no-deps", dst)
        print("installed prebuilt wheel (no compile); cached to Drive.")
    except Exception as e:
        if os.path.exists(dst):
            os.remove(dst)
        print("prebuilt fetch failed:", e, "\nfalling back to SOURCE BUILD (slow, one time)...")
        _pip(f"mamba-ssm=={MAMBA_VER}", "--no-build-isolation")


ensure_mamba()

# smoke check: Mamba-3 forward pass on this GPU
from mamba_ssm.modules.mamba3 import Mamba3
_dt = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
m = Mamba3(d_model=256, layer_idx=0, device='cuda', dtype=_dt)
x = torch.randn(2, 64, 256, device='cuda', dtype=_dt)
print('Mamba-3 forward OK:', m(x).shape)

In [ ]:
# === 3. Tokenizer (once) ===
# HF_TOKEN / timeout env vars are set in cell 1. Do NOT hard-code a token here.
import os
if not os.path.exists(f'{DATA}/bn_bpe32k.model'):
    !python scripts/train_tokenizer.py --out_prefix $DATA/bn_bpe32k --sample_docs 400000
else:
    print('tokenizer exists, skipping')

In [ ]:
# === 4. Pretokenize (once; restartable) ===
# Train tokens: set AFTER profiling (cell 6). Start with 1.2B; raise to 2.5B if throughput allows.
!python scripts/pretokenize.py --sp_model $DATA/bn_bpe32k.model --out_dir $DATA --split train --target_tokens 1200000000
!python scripts/pretokenize.py --sp_model $DATA/bn_bpe32k.model --out_dir $DATA --split test  --target_tokens 30000000

In [ ]:
# === 5. Parameter match check ===
!python scripts/count_params.py configs/transformer_53m.json configs/mamba3_53m.json

In [ ]:
# === 6. Throughput profiling -> lock the token budget (plan §6) ===
!python -m bangla_ssm.profile_throughput --config configs/transformer_53m.json --batch_size 32 --grad_accum 2 --steps 30
!python -m bangla_ssm.profile_throughput --config configs/mamba3_53m.json --batch_size 32 --grad_accum 2 --steps 30

In [ ]:
# === 7. Toy correctness runs (both archs, ~200 steps) ===
!python -m bangla_ssm.train --config configs/transformer_toy.json --data_dir $DATA --out_dir $RUNS/toy_tf    --token_budget 50000000 --batch_size 16 --max_steps 200
!python -m bangla_ssm.train --config configs/mamba3_toy.json      --data_dir $DATA --out_dir $RUNS/toy_m3    --token_budget 50000000 --batch_size 16 --max_steps 200
# Loss should drop from ~10.4 (ln 32768) to well under 7 within 200 steps for both.

In [ ]:
# === 8. MAIN RUNS (re-run this cell after any disconnect; it resumes) ===
TOKEN_BUDGET = 1_000_000_000   # <-- lock after profiling
!python -m bangla_ssm.train --config configs/transformer_53m.json --data_dir $DATA --out_dir $RUNS/tf_53m --token_budget $TOKEN_BUDGET --batch_size 32 --grad_accum 2
!python -m bangla_ssm.train --config configs/mamba3_53m.json      --data_dir $DATA --out_dir $RUNS/m3_53m --token_budget $TOKEN_BUDGET --batch_size 32 --grad_accum 2

In [ ]:
# === 9. Probes: generate + (optional) validate on an existing Bangla model ===
!python -m bangla_ssm.probes.generate --out_dir $DATA/probes
# Probe validation on an off-the-shelf model (plan §4.4):
# !python -m bangla_ssm.eval_minimal_pairs --tsv $DATA/probes/sva.tsv --hf_model flax-community/gpt2-bengali --out $RUNS/probe_check_gpt2bn.json

In [ ]:
# === 10. Final evaluation ===
import glob
tf_ck = sorted(glob.glob(f'{RUNS}/tf_53m/ckpt_*.pt'))[-1]
m3_ck = sorted(glob.glob(f'{RUNS}/m3_53m/ckpt_*.pt'))[-1]
print(tf_ck, m3_ck)

for name, cfg, ck in [('tf', 'configs/transformer_53m.json', tf_ck), ('m3', 'configs/mamba3_53m.json', m3_ck)]:
    !python -m bangla_ssm.eval_ppl --config {cfg} --ckpt {ck} --data_dir $DATA --limit_tokens 20000000
    !python -m bangla_ssm.eval_minimal_pairs --tsv $DATA/probes/sva.tsv $DATA/probes/attraction.tsv $DATA/probes/honorific.tsv $DATA/probes/discourse.tsv --config {cfg} --ckpt {ck} --sp_model $DATA/bn_bpe32k.model --out $RUNS/results_{name}.json

## Extension: hybrid model + second seed

Motivated by the seed-1 result (attention wins *local* agreement; Mamba-3 wins
*distance/discourse* and perplexity), we add a **hybrid** — a Mamba-3 backbone
with attention at 2 of 15 layers (Samba/Jamba-style) — and run a **second seed**
for all three models so the comparison has variance bars.

Run cells 12→16 in order. Everything reuses the same data, tokenizer, and probes.
Total added compute ≈ 4–6 GPU-hours on the A100.

In [ ]:
# === 12. Param-match the hybrid to ~24.5M non-embedding ===
# Should print OK (<2%). If it says ADJUST, nudge n_layer in configs/hybrid_53m.json.
!python scripts/count_params.py configs/transformer_53m.json configs/hybrid_53m.json

In [ ]:
# === 13. Hybrid toy sanity (loss should start ~10.4 and fall, like the others) ===
!rm -rf $RUNS/toy_hybrid
!python -m bangla_ssm.train --config configs/hybrid_toy.json --data_dir $DATA --out_dir $RUNS/toy_hybrid --token_budget 50000000 --batch_size 16 --max_steps 200

In [ ]:
# === 14. Hybrid main run (seed 1), same 1B-token budget; resumable ===
!python -m bangla_ssm.train --config configs/hybrid_53m.json --data_dir $DATA --out_dir $RUNS/hybrid_53m --token_budget 1000000000 --batch_size 32 --grad_accum 2

In [ ]:
# === 15. Second seed (2024) for ALL THREE models -> variance ===
# Each run is resumable; re-run this cell after a disconnect.
for name, cfg in [('tf_53m', 'configs/transformer_53m.json'),
                  ('m3_53m', 'configs/mamba3_53m.json'),
                  ('hybrid_53m', 'configs/hybrid_53m.json')]:
    !python -m bangla_ssm.train --config {cfg} --data_dir $DATA --out_dir $RUNS/{name}_s2 --token_budget 1000000000 --batch_size 32 --grad_accum 2 --seed 2024

In [ ]:
# === 16. Evaluate EVERY trained checkpoint (ppl + probes) -> results_*.json ===
import glob
runs = {
    'tf_s1':     ('configs/transformer_53m.json', f'{RUNS}/tf_53m'),
    'm3_s1':     ('configs/mamba3_53m.json',      f'{RUNS}/m3_53m'),
    'hybrid_s1': ('configs/hybrid_53m.json',      f'{RUNS}/hybrid_53m'),
    'tf_s2':     ('configs/transformer_53m.json', f'{RUNS}/tf_53m_s2'),
    'm3_s2':     ('configs/mamba3_53m.json',      f'{RUNS}/m3_53m_s2'),
    'hybrid_s2': ('configs/hybrid_53m.json',      f'{RUNS}/hybrid_53m_s2'),
}
probes = ' '.join(f'{DATA}/probes/{p}.tsv' for p in ['sva', 'attraction', 'honorific', 'discourse'])
for tag, (cfg, outdir) in runs.items():
    cks = sorted(glob.glob(f'{outdir}/ckpt_*.pt'))
    if not cks:
        print('skip (no checkpoint yet):', tag); continue
    ck = cks[-1]
    print(f'=== {tag}  {ck}')
    !python -m bangla_ssm.eval_ppl --config {cfg} --ckpt {ck} --data_dir $DATA --limit_tokens 20000000
    !python -m bangla_ssm.eval_minimal_pairs --tsv {probes} --config {cfg} --ckpt {ck} --sp_model $DATA/bn_bpe32k.model --out $RUNS/results_{tag}.json

## Publish to Hugging Face Hub

Pushes the probe suite (as a dataset) and the 6 trained checkpoints + configs +
eval results (as a model repo) directly from Drive — no local download needed.

**Requires a WRITE-scoped HF token.** If the `HF_TOKEN` Colab Secret was created
only for reading (downloading FineWeb-2 earlier), edit that secret's value to a
token with **write** access from https://huggingface.co/settings/tokens, or the
uploads below will fail with a 403.

In [ ]:
# === 17. Push probe suite to Hugging Face Hub as a dataset ===
from huggingface_hub import HfApi, login
import os

login(token=os.environ["HF_TOKEN"])
api = HfApi()
HF_USER = api.whoami()["name"]
print("logged in as:", HF_USER)

DATASET_REPO = f"{HF_USER}/bangla-agreement-probes"
api.create_repo(DATASET_REPO, repo_type="dataset", exist_ok=True, private=False)

for name in ["sva", "attraction", "honorific", "discourse"]:
    api.upload_file(
        path_or_fileobj=f"{DATA}/probes/{name}.tsv",
        path_in_repo=f"{name}.tsv",
        repo_id=DATASET_REPO,
        repo_type="dataset",
    )

dataset_readme = f"""---
license: mit
language:
- bn
---

# Bangla agreement & honorific-register minimal pairs

4,790 hand-built minimal pairs probing Bangla subject-verb person/honorific
agreement (distance-binned: none/short/medium/long), agreement attraction,
and cross-sentence pro-drop register agreement. Native-speaker reviewed.
Built for a controlled Transformer vs Mamba-3 vs Hybrid case study.

| File | Phenomenon | Pairs |
|---|---|---|
| sva.tsv | subject-verb agreement, distance-binned | 3300 |
| attraction.tsv | agreement attraction (competing-person lure) | 1190 |
| honorific.tsv | honorific-register agreement | 210 |
| discourse.tsv | pro-drop register agreement across a sentence boundary | 90 |

Columns: `id, phenomenon, subj_person, wrong_person, lure_person, distance, tense, sen, wrong_sen`.
Scoring: a model is correct on a pair if log-prob(`sen`) > log-prob(`wrong_sen`).

Code + generator: https://github.com/sahilaf/Mamba3_Bangla_Case_Study
"""
api.upload_file(
    path_or_fileobj=dataset_readme.encode("utf-8"),
    path_in_repo="README.md",
    repo_id=DATASET_REPO,
    repo_type="dataset",
)
print(f"dataset live at https://huggingface.co/datasets/{DATASET_REPO}")

In [ ]:
# === 18. Push checkpoints + configs + eval results to Hugging Face Hub ===
import glob, os, torch
from huggingface_hub import HfApi

api = HfApi()
HF_USER = api.whoami()["name"]
MODEL_REPO = f"{HF_USER}/mamba3-bangla-case-study"
api.create_repo(MODEL_REPO, repo_type="model", exist_ok=True, private=False)

# Upload only model weights (drop optimizer state) -> ~1/3 the file size;
# optimizer state is only needed to resume training, not to reproduce eval.
def extract_weights(ckpt_path, tmp_path):
    state = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    torch.save({"model": state["model"], "step": state["step"]}, tmp_path)
    return tmp_path

run_dirs = {
    "transformer_seed1": ("configs/transformer_53m.json", f"{RUNS}/tf_53m"),
    "mamba3_seed1":      ("configs/mamba3_53m.json",      f"{RUNS}/m3_53m"),
    "hybrid_seed1":      ("configs/hybrid_53m.json",      f"{RUNS}/hybrid_53m"),
    "transformer_seed2": ("configs/transformer_53m.json", f"{RUNS}/tf_53m_s2"),
    "mamba3_seed2":      ("configs/mamba3_53m.json",      f"{RUNS}/m3_53m_s2"),
    "hybrid_seed2":      ("configs/hybrid_53m.json",      f"{RUNS}/hybrid_53m_s2"),
}

for tag, (cfg_path, out_dir) in run_dirs.items():
    cks = sorted(glob.glob(f"{out_dir}/ckpt_*.pt"))
    if not cks:
        print("skip (no checkpoint):", tag); continue
    tmp = f"/content/{tag}_weights.pt"
    extract_weights(cks[-1], tmp)
    api.upload_file(path_or_fileobj=tmp, path_in_repo=f"{tag}/checkpoint.pt", repo_id=MODEL_REPO)
    api.upload_file(path_or_fileobj=cfg_path, path_in_repo=f"{tag}/config.json", repo_id=MODEL_REPO)
    os.remove(tmp)
    print("uploaded", tag)

for f in glob.glob(f"{RUNS}/results_*.json"):
    api.upload_file(path_or_fileobj=f, path_in_repo=f"results/{os.path.basename(f)}", repo_id=MODEL_REPO)
for f in [f"{DATA}/bn_bpe32k.model", f"{DATA}/bn_bpe32k.vocab"]:
    if os.path.exists(f):
        api.upload_file(path_or_fileobj=f, path_in_repo=f"tokenizer/{os.path.basename(f)}", repo_id=MODEL_REPO)

model_readme = f"""---
license: mit
language:
- bn
---

# Mamba-3 vs Transformer vs Hybrid — Bangla case study checkpoints

Three ~24.5M non-embedding-parameter causal LMs trained from scratch on 1B
tokens of Bangla (FineWeb-2 `ben_Beng`), 2 seeds each:
- a Llama-style Transformer (RoPE, SwiGLU, full attention)
- an official Mamba-3 tower ([state-spaces/mamba](https://github.com/state-spaces/mamba))
- a hybrid: Mamba-3 backbone with attention at 2 of 15 layers

**Headline finding:** the Transformer's subject-verb agreement accuracy degrades
as the distance between subject and verb grows; Mamba-3's does not (holds across
both seeds). Transformer still leads on raw agreement/honorific accuracy;
Mamba-3 leads on perplexity and the hardest (cross-sentence) discourse probe.

See `results/results_*.json` for full held-out perplexity and per-probe,
per-condition accuracy for every checkpoint. `checkpoint.pt` contains only
model weights (no optimizer state) — load with `bangla_ssm.models.build_model`
using the matching `config.json`.

Code + probe generator: https://github.com/sahilaf/Mamba3_Bangla_Case_Study
Probe dataset: https://huggingface.co/datasets/{HF_USER}/bangla-agreement-probes
"""
api.upload_file(path_or_fileobj=model_readme.encode("utf-8"), path_in_repo="README.md", repo_id=MODEL_REPO)
print(f"model repo live at https://huggingface.co/{MODEL_REPO}")

## Revision: statistical rigor + ablations (reviewer response)

These cells address the main reviewer concerns. **All optional** — the paper's
Wilson CIs and the headline finding stand without them, but these strengthen it:

- **Cell 20** — 3 extra seeds (5 total) per model → answers "only 2 seeds".
- **Cell 21** — hybrid ablation (1 vs 2 vs 4 attention layers) → answers "why 2?".
- **Cell 22** — per-pair dumps + **McNemar paired significance tests**, plus Wilson CIs.

Compute: cell 20 is ~14 GPU-hr (run across sessions; resumable), cell 21 ~4 GPU-hr,
cell 22 ~15 min. After running, paste the new numbers back to regenerate figures/tables.

In [ ]:
# === 20. Extra seeds (3419, 5150, 8888) for all three main models -> 5 seeds total ===
# Addresses the "only 2 seeds" reviewer concern. Each run resumes if interrupted.
# ~1.5-1.8 GPU-hr per run x 9 runs; run overnight or across sessions.
for seed in [3419, 5150, 8888]:
    for name, cfg in [('tf_53m', 'configs/transformer_53m.json'),
                      ('m3_53m', 'configs/mamba3_53m.json'),
                      ('hybrid_53m', 'configs/hybrid_53m.json')]:
        !python -m bangla_ssm.train --config {cfg} --data_dir $DATA --out_dir $RUNS/{name}_s{seed} --token_budget 1000000000 --batch_size 32 --grad_accum 2 --seed {seed}
        !python -m bangla_ssm.eval_minimal_pairs --tsv $DATA/probes/sva.tsv $DATA/probes/attraction.tsv $DATA/probes/honorific.tsv $DATA/probes/discourse.tsv --config {cfg} --ckpt $(ls -t $RUNS/{name}_s{seed}/ckpt_*.pt | head -1) --sp_model $DATA/bn_bpe32k.model --out $RUNS/results_{name}_s{seed}.json

In [ ]:
# === 21. Hybrid attention-count ablation (why 2 layers?) ===
# Trains 1-attn and 4-attn hybrids (seed 1) to compare against the 2-attn main.
!python scripts/count_params.py configs/transformer_53m.json configs/hybrid_1attn_53m.json
!python scripts/count_params.py configs/transformer_53m.json configs/hybrid_4attn_53m.json
for name, cfg in [('hybrid_1attn', 'configs/hybrid_1attn_53m.json'),
                  ('hybrid_4attn', 'configs/hybrid_4attn_53m.json')]:
    !python -m bangla_ssm.train --config {cfg} --data_dir $DATA --out_dir $RUNS/{name} --token_budget 1000000000 --batch_size 32 --grad_accum 2
    !python -m bangla_ssm.eval_ppl --config {cfg} --ckpt $(ls -t $RUNS/{name}/ckpt_*.pt | head -1) --data_dir $DATA --limit_tokens 20000000
    !python -m bangla_ssm.eval_minimal_pairs --tsv $DATA/probes/sva.tsv $DATA/probes/attraction.tsv $DATA/probes/honorific.tsv $DATA/probes/discourse.tsv --config {cfg} --ckpt $(ls -t $RUNS/{name}/ckpt_*.pt | head -1) --sp_model $DATA/bn_bpe32k.model --out $RUNS/results_{name}.json

In [ ]:
# === 22. Per-pair dump + significance testing (McNemar) ===
# Re-scores every checkpoint, dumping per-pair correctness, then runs paired
# McNemar tests. Wilson CIs come from paper/stats.py directly (no data needed).
import glob
DUMPS = f'{RUNS}/per_pair'
probes = ' '.join(f'{DATA}/probes/{p}.tsv' for p in ['sva', 'attraction', 'honorific', 'discourse'])
ckpts = {
    'tf_s1': ('configs/transformer_53m.json', f'{RUNS}/tf_53m'),
    'm3_s1': ('configs/mamba3_53m.json',      f'{RUNS}/m3_53m'),
    'hybrid_s1': ('configs/hybrid_53m.json',  f'{RUNS}/hybrid_53m'),
}
for tag, (cfg, outdir) in ckpts.items():
    cks = sorted(glob.glob(f'{outdir}/ckpt_*.pt'))
    if not cks:
        print('skip', tag); continue
    !python -m bangla_ssm.eval_minimal_pairs --tsv {probes} --config {cfg} --ckpt {cks[-1]} --sp_model $DATA/bn_bpe32k.model --dump_per_pair {DUMPS} --dump_tag {tag}
print('\n--- Wilson CIs + McNemar ---')
!python paper/stats.py --dumps {DUMPS}